In [1]:
import json
import os

import numpy as np

from Pipeline.Data.GallstoneDataSet import GallstoneDataSet
from Pipeline.Model.EvaluationELM import EvaluationELM
from Pipeline.Model.ExtremeLearningMachine import sigmoid

In [2]:
gallstone_dataset = GallstoneDataSet()
gallstone_dataset.fetch_data_path_1()

features_size = gallstone_dataset.x_train_scaled.shape[1]

In [3]:
hidden_size_explore = range(55, 59)
lambda_value_explore = 1.5 ** np.arange(-2, 2)

In [4]:
# 1. Instantiate the Evaluator Object ONCE
evaluator = EvaluationELM(
    x=gallstone_dataset.x_train,
    y=gallstone_dataset.y_train,
    activation_function=sigmoid,
    elm_initial_state_range= range(41, 45),
    data_split_state_range= range(1, 4),
    test_size=0.2,
    k_fold=5
)

In [5]:
combination_record , combination_result = evaluator.grid_search_hidden_size_and_lambda(
    hidden_size_range=hidden_size_explore,
    lambda_range=lambda_value_explore
)

In [6]:
combination_result_list = EvaluationELM.extract_top_results(combination_result)
best_combination_result = combination_result_list.iloc[0]

In [7]:
dataset_dir = '../../Dataset/JSON/'
config_file = os.path.join(dataset_dir, 'full_model_configs.json')

with open(config_file, 'r') as f:
    saved_configs = json.load(f)

first_model_config = saved_configs[0]

best_hidden_size = first_model_config["Hidden_Nodes"]
best_lambda_value = first_model_config["Lambda_Value"]

print(f"Retrieved Hidden Nodes: {best_hidden_size}")

for config in saved_configs:
    model_type = config["Model_Types"]
    h_nodes = config["Hidden_Nodes"]
    print(f"Model: {model_type} -> Hidden Nodes: {h_nodes}")

Retrieved Hidden Nodes: 58
Model: Best_Hidden_Nodes -> Hidden Nodes: 58
Model: Best_Lambda -> Hidden Nodes: 58
Model: Best_Size_and_Lambda -> Hidden Nodes: 58


In [8]:
hidden_size_smaller = range(features_size//3, best_hidden_size)
lambda_value_wider = 2.0 ** np.arange(-10, 11)

optimization_record , optimization_result = evaluator.grid_search_hidden_size_and_lambda(
    hidden_size_range=hidden_size_smaller,
    lambda_range=lambda_value_wider
)

In [9]:
optimization_result_list = EvaluationELM.extract_top_results(optimization_result)
best_optimization_result = optimization_result_list.iloc[0]

In [10]:
new_configs_payload = [
    {
        "Model_Types" : "Grid_Combination",
        "Hidden_Nodes": int(best_combination_result['Hidden_Nodes']),
        "Activation": "sigmoid",
        "Lambda_Value": float(best_combination_result['Lambda_Value'])
    },
    {
        "Model_Types" : "Grid_Optimization",
        "Hidden_Nodes": int(best_optimization_result['Hidden_Nodes']),
        "Activation": "sigmoid",
        "Lambda_Value": float(best_optimization_result['Lambda_Value'])
    }
]

In [11]:
if os.path.exists(config_file):
    with open(config_file, 'r') as f:
        existing_configs = json.load(f)
else:
    existing_configs = []

existing_configs.extend(new_configs_payload)

with open(config_file, 'w') as f:
    json.dump(existing_configs, f, indent=4)

print(f"Successfully appended to {config_file}. Total configs now: {len(existing_configs)}")

Successfully appended to ../../Dataset/JSON/full_model_configs.json. Total configs now: 5
